# 📖 Notebook 3: Real-Time Notifications

When two users match on Tinder, **both need to know immediately**:
- **Person B** (the second swiper) sees a "You Matched!" screen right away — easy, since they just triggered the match.
- **Person A** (the first swiper) might have swiped days ago and could be offline. They need a push notification.

This notebook explores how to deliver real-time match notifications using **Redis Pub/Sub** for online users and discusses the push notification pattern (APNS/FCM) for offline users.

## Learning Objectives

By the end of this notebook, you'll understand:
- How Redis Pub/Sub works for real-time message delivery
- The difference between online (Pub/Sub) and offline (push) notification patterns
- How to store notifications for later retrieval
- How the complete match notification flow works end-to-end

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/tinder
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `tinder_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import time
import json
import threading

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "tinder_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

def query(sql, params=None):
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute(sql, params)
    rows = cursor.fetchall()
    conn.close()
    return rows

def execute(sql, params=None):
    conn = get_db()
    conn.autocommit = True
    cursor = conn.cursor()
    cursor.execute(sql, params)
    conn.close()

# Test connections
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 The Notification Challenge

When a match happens, we need to notify two people in very different situations:

```
┌────────────────────────────────────────────────────────────┐
│                     MATCH HAPPENS                          │
├─────────────────────────┬──────────────────────────────────┤
│  Person B (2nd swiper)  │  Person A (1st swiper)           │
│  ✅ Online right now    │  ❓ Maybe online, maybe not      │
│  → Show match screen    │  → Need to reach them somehow    │
│    immediately          │                                  │
├─────────────────────────┼──────────────────────────────────┤
│  EASY: they just        │  If ONLINE: Redis Pub/Sub        │
│  triggered the match,   │  If OFFLINE: Push notification   │
│  return it in the       │    (APNS for iOS,                │
│  swipe response         │     FCM for Android)             │
└─────────────────────────┴──────────────────────────────────┘
```

Person B is trivial — the swipe API response just includes `{"match": true}`.  
Person A is the interesting challenge. Let's solve it with Redis Pub/Sub.

## 📡 Redis Pub/Sub: How It Works

Redis Pub/Sub is a **publish-subscribe messaging system** built into Redis:

- **Subscribers** listen on a channel (like tuning into a radio station)
- **Publishers** send messages to a channel (like broadcasting)
- Messages are delivered **instantly** to all current subscribers
- Messages are **NOT stored** — if nobody is listening, the message is lost ("fire and forget")

For Tinder, each online user subscribes to their own channel: `notifications:user:{id}`

```
User 4 (Mia) opens app → SUBSCRIBE notifications:user:4
Match happens           → PUBLISH notifications:user:4 "You matched with Noah!"
Mia's app               → Receives message instantly
```

### ⚠️ Pub/Sub trade-offs you must know

| Property | Pub/Sub behaviour |
|---|---|
| Delivery | At-most-once (lost if nobody is subscribed) |
| Persistence | None — messages live only in RAM during delivery |
| Ordering | Per-channel, per-publisher |
| Scale across Redis nodes | Only within the same Redis (or Cluster `SSUBSCRIBE` for sharded Pub/Sub) |

**That's why we always save notifications to PostgreSQL first.** The DB is the durable source of truth; Pub/Sub is a best-effort "you might get a real-time ping" delivery channel.

### When Pub/Sub isn't enough

If you need **durable** real-time messaging (replay messages, consumer groups, guaranteed delivery), reach for:
- **Redis Streams** (`XADD` / `XREAD`) — durable log with consumer groups, still inside Redis
- **Kafka** or **RabbitMQ** — full-blown message brokers for bigger systems
- **WebSocket + message store** — keep a per-user queue in PostgreSQL / DynamoDB and push via WebSocket

Let's see basic Pub/Sub in action:


In [ ]:
# Basic Redis Pub/Sub demo
# We'll use a background thread as the subscriber (simulating a user's app)

received_messages = []  # Collect messages received by the subscriber

def wait_for_subscribers(channel: str, expected: int, timeout: float = 5.0) -> bool:
    """Block until `channel` has `expected` subscribers (or give up).

    Publishing before the SUBSCRIBE has reached the server silently drops the
    message — Pub/Sub has no buffer. Polling PUBSUB NUMSUB is deterministic;
    a `time.sleep(0.5)` guess is not.
    """
    deadline = time.time() + timeout
    while time.time() < deadline:
        if dict(r.pubsub_numsub(channel))[channel] >= expected:
            return True
        time.sleep(0.02)
    return False

def user_app_listener(user_id: int, duration: float = 3.0):
    """Simulate a user's app listening for notifications.
    Uses a SEPARATE Redis connection (required for Pub/Sub)."""
    listener_redis = redis.Redis(**REDIS_CONFIG)
    pubsub = listener_redis.pubsub(ignore_subscribe_messages=True)
    channel = f"notifications:user:{user_id}"
    
    pubsub.subscribe(channel)
    print(f"   📱 User {user_id}'s app is now listening on '{channel}'")
    
    # Poll with a deadline rather than iterating pubsub.listen(): listen() blocks
    # forever once the last message has arrived, which would leave this thread
    # (and its Redis connection) hanging around for the life of the kernel.
    deadline = time.time() + duration
    while time.time() < deadline:
        message = pubsub.get_message(timeout=0.1)
        if message and message['type'] == 'message':
            data = json.loads(message['data'])
            received_messages.append(data)
            print(f"   🔔 User {user_id} received: {data['message']}")
            break
    
    pubsub.unsubscribe()
    pubsub.close()
    listener_redis.close()

# Start Mia's app listening in the background
print("📡 Redis Pub/Sub Demo\n")
listener_thread = threading.Thread(target=user_app_listener, args=(4, 3.0), daemon=True)
listener_thread.start()
assert wait_for_subscribers("notifications:user:4", 1), (
    "Mia's app never showed up as a subscriber — publishing now would throw the "
    "message away, and the demo would 'pass' while proving nothing."
)

# Now publish a message (simulating a match notification)
notification = {
    "type": "match",
    "message": "🎉 You matched with Noah Thompson!",
    "match_user_id": 14,
    "match_user_name": "Noah Thompson",
    "timestamp": time.time()
}

print(f"   📤 Publishing notification to notifications:user:4")
delivered_to = r.publish("notifications:user:4", json.dumps(notification))

# Wait for listener to receive and finish
listener_thread.join(timeout=5)
print()
print(f"✅ Message delivered in real-time via Redis Pub/Sub!")

assert delivered_to == 1, f"expected exactly 1 subscriber to receive it, got {delivered_to}"
assert not listener_thread.is_alive(), "the subscriber thread should have exited on its own"
assert len(received_messages) == 1, (
    f"Mia's app should have received exactly one message, got {len(received_messages)}"
)
assert received_messages[0]['message'] == notification['message']


## 🏗️ Building the Complete Notification System

A production notification system needs to handle both online and offline users. Here's the full flow:

```
Match detected
    │
    ├── 1. Save notification to PostgreSQL (permanent record)
    │
    ├── 2. Try Redis Pub/Sub (for online users)
    │      ├── Delivered? → Done! User sees it instantly
    │      └── No listeners? → Message is lost (that's OK)
    │
    └── 3. Send push notification (for offline/background users)
           ├── iOS → Apple Push Notification Service (APNS)
           └── Android → Firebase Cloud Messaging (FCM)
```

The key insight: we **always** save to PostgreSQL first. Redis Pub/Sub is fire-and-forget — if the user isn't listening, the message vanishes. The database record ensures they can always see their notifications when they open the app.

In [ ]:
class NotificationService:
    """Handles match notifications for both online and offline users."""
    
    def __init__(self):
        self.redis = get_redis()
    
    def send_match_notification(self, user_id: int, match_user_id: int, match_id: int):
        """Send a match notification to a user."""
        
        # Get the matched user's name for the notification message
        match_user = query("SELECT name FROM users WHERE id = %s", (match_user_id,))[0]
        message = f"🎉 You matched with {match_user['name']}!"
        
        # Step 1: ALWAYS save to PostgreSQL (durable record)
        execute(
            "INSERT INTO notifications (user_id, type, message, match_id) VALUES (%s, %s, %s, %s)",
            (user_id, 'match', message, match_id)
        )
        print(f"   💾 Saved notification to PostgreSQL for user {user_id}")
        
        # Step 2: Try real-time delivery via Redis Pub/Sub
        notification_data = json.dumps({
            "type": "match",
            "message": message,
            "match_user_id": match_user_id,
            "match_id": match_id,
            "timestamp": time.time()
        })
        
        channel = f"notifications:user:{user_id}"
        listeners = self.redis.publish(channel, notification_data)
        
        if listeners > 0:
            print(f"   📡 Real-time delivery via Pub/Sub ({listeners} listener(s))")
        else:
            print(f"   📱 User {user_id} is offline — would send push notification")
            # In production: call APNS/FCM here
            # self._send_push_notification(user_id, message)
        
        return listeners > 0
    
    def get_unread_notifications(self, user_id: int):
        """Get all unread notifications for a user (when they open the app)."""
        return query("""
            SELECT id, type, message, match_id, created_at
            FROM notifications
            WHERE user_id = %s AND is_read = FALSE
            ORDER BY created_at DESC
        """, (user_id,))
    
    def mark_as_read(self, notification_ids: list):
        """Mark notifications as read."""
        if not notification_ids:
            return
        placeholders = ','.join(['%s'] * len(notification_ids))
        execute(
            f"UPDATE notifications SET is_read = TRUE WHERE id IN ({placeholders})",
            tuple(notification_ids)
        )

notifier = NotificationService()
print("✅ NotificationService ready")
print("   - Saves to PostgreSQL (durable)")
print("   - Publishes via Redis Pub/Sub (real-time)")
print("   - Falls back to push notification (offline)")

## 🎬 End-to-End Demo: The Complete Match Flow

Let's put it all together — from swipe to match to notification. We'll simulate:
1. **Mia** (User 4) is online and listening for notifications
2. **Noah** (User 14) opens the app and swipes right on Mia
3. Mia already swiped right on Noah (from seed data) → **Match!**
4. Both get notified

In [ ]:
# The complete swipe handler with notifications

# Re-create the atomic swipe infrastructure from Notebook 2 — the idempotent
# version, because notifications are exactly what you must not send twice.
SWIPE_SCRIPT = """
local previous = redis.call('HGET', KEYS[1], ARGV[1])
redis.call('HSET', KEYS[1], ARGV[1], ARGV[2])
redis.call('EXPIRE', KEYS[1], 2592000)
local other = redis.call('HGET', KEYS[1], ARGV[3])
if other == false then other = '' end
local replay = 0
if previous == ARGV[2] then replay = 1 end
return {other, replay}
"""
swipe_sha = r.script_load(SWIPE_SCRIPT)

def get_swipe_key(user_a, user_b):
    smaller, larger = sorted([user_a, user_b])
    return f"swipe:{smaller}:{larger}"

def full_swipe_handler(swiper_id: int, target_id: int, direction: str):
    """Complete swipe flow: atomic match detection + DB persistence + notifications."""
    
    swiper = query("SELECT name FROM users WHERE id = %s", (swiper_id,))[0]
    target = query("SELECT name FROM users WHERE id = %s", (target_id,))[0]
    
    # Step 1: Atomic swipe + match check in Redis (replays report no match)
    key = get_swipe_key(swiper_id, target_id)
    other_swipe, replay = r.evalsha(
        swipe_sha, 1, key,
        f"{swiper_id}_swipe", direction, f"{target_id}_swipe"
    )
    is_match = (not replay) and direction == 'right' and other_swipe == 'right'
    
    print(f"   ⚡ Redis: {swiper['name']} swipes {direction} on {target['name']}")
    
    # Step 2: Persist swipe to PostgreSQL
    execute(
        "INSERT INTO swipes (swiper_id, target_id, direction) VALUES (%s, %s, %s) ON CONFLICT DO NOTHING",
        (swiper_id, target_id, direction)
    )
    print(f"   💾 Swipe saved to PostgreSQL")
    
    # Step 3: If match, persist and notify
    if is_match:
        user1, user2 = sorted([swiper_id, target_id])
        # `ON CONFLICT DO NOTHING ... RETURNING id` gives back a row ONLY when the
        # INSERT really happened. That is our second idempotency guard: if the match
        # row was already there, somebody already told these two — sending again
        # would be a duplicate "It's a match!" push for an old match.
        conn = get_db()
        conn.autocommit = True
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO matches (user1_id, user2_id) VALUES (%s, %s) "
            "ON CONFLICT DO NOTHING RETURNING id",
            (user1, user2)
        )
        inserted = cursor.fetchone()
        conn.close()
        
        if inserted is None:
            existing = query(
                "SELECT id FROM matches WHERE user1_id = %s AND user2_id = %s",
                (user1, user2)
            )
            print(f"\n   ↩️  Match #{existing[0]['id']} already existed — "
                  f"skipping duplicate notifications")
            return is_match
        
        match_id = inserted[0]
        
        print(f"\n   🎉 IT'S A MATCH! {swiper['name']} ↔ {target['name']}")
        print(f"   💾 Match #{match_id} saved to PostgreSQL")
        print()
        
        # Notify Person B (swiper) — they get it in the API response
        print(f"   📬 Notifying {swiper['name']} (swiper — via API response):")
        notifier.send_match_notification(swiper_id, target_id, match_id)
        
        # Notify Person A (target) — they need Pub/Sub or push
        print(f"   📬 Notifying {target['name']} (first swiper — via Pub/Sub):")
        notifier.send_match_notification(target_id, swiper_id, match_id)
    
    return is_match

print("✅ Full swipe handler ready (swipe → match → notify)")

In [ ]:
# End-to-end demo!

# Reset the state for this pair so the demo is repeatable: the Redis swipe hash,
# Noah's swipe row, the match, and any notifications that point at it. (Order
# matters — notifications hold a foreign key into matches.)
r.delete(get_swipe_key(4, 14))
execute("""
    DELETE FROM notifications
    WHERE match_id IN (SELECT id FROM matches WHERE user1_id = 4 AND user2_id = 14)
""")
execute("DELETE FROM matches WHERE user1_id = 4 AND user2_id = 14")
execute("DELETE FROM swipes WHERE swiper_id = 14 AND target_id = 4")

# Pre-load Mia's existing right swipe on Noah into Redis
r.evalsha(swipe_sha, 1, get_swipe_key(4, 14),
          "4_swipe", "right", "14_swipe")

print("=" * 65)
print("   END-TO-END DEMO: Noah swipes on Mia")
print("=" * 65)
print()

# Start Mia's app listening for notifications
mia_notifications = []

def mia_listener(duration: float = 3.0):
    lr = redis.Redis(**REDIS_CONFIG)
    ps = lr.pubsub(ignore_subscribe_messages=True)
    ps.subscribe("notifications:user:4")
    deadline = time.time() + duration
    while time.time() < deadline:
        msg = ps.get_message(timeout=0.1)
        if msg and msg['type'] == 'message':
            mia_notifications.append(json.loads(msg['data']))
            break
    ps.unsubscribe()
    ps.close()
    lr.close()

print("📱 Mia(4) opens the app and starts listening...")
mia_thread = threading.Thread(target=mia_listener, daemon=True)
mia_thread.start()
assert wait_for_subscribers("notifications:user:4", 1), "Mia's app never subscribed"

# Noah swipes right on Mia
print("\n📱 Noah(14) swipes RIGHT on Mia(4)...\n")
match = full_swipe_handler(14, 4, 'right')

# Wait for Mia's listener to receive
mia_thread.join(timeout=5)

print()
print("=" * 65)
print("   RESULTS")
print("=" * 65)
print(f"   Match detected: {match}")
print(f"   Mia received {len(mia_notifications)} real-time notification(s)")
if mia_notifications:
    print(f"   Message: {mia_notifications[0]['message']}")

stored = query("""
    SELECT n.user_id, n.message
    FROM notifications n
    JOIN matches m ON m.id = n.match_id
    WHERE m.user1_id = 4 AND m.user2_id = 14
    ORDER BY n.user_id
""")

assert match, "Noah's right swipe on a profile that already liked him must match"
assert len(mia_notifications) == 1, (
    f"Mia was subscribed, so Pub/Sub should have delivered exactly one message; "
    f"got {len(mia_notifications)}"
)
assert {row['user_id'] for row in stored} == {4, 14}, (
    f"Both sides of the match need a durable notification row (Pub/Sub is "
    f"best-effort); found rows for {sorted(row['user_id'] for row in stored)}"
)

# Now the part that used to double-notify: Noah's phone retries the same swipe.
print()
print("📱 Noah's phone lost the response and retries the SAME swipe...\n")
retry_match = full_swipe_handler(14, 4, 'right')

after = query("""
    SELECT n.id
    FROM notifications n
    JOIN matches m ON m.id = n.match_id
    WHERE m.user1_id = 4 AND m.user2_id = 14
""")
print()
print(f"   Retry reported a match: {retry_match}")
print(f"   Notification rows for this match: {len(stored)} → {len(after)}")

assert retry_match is False, "a replayed swipe must not report a second match"
assert len(after) == len(stored), (
    f"A retried swipe must not create duplicate notifications: {len(stored)} rows "
    f"became {len(after)}. Two 'It's a match!' pushes for one match is a bug users "
    f"actually notice."
)


## 📬 Offline Users: Catching Up on Notifications

When a user opens the app after being offline, they need to see any notifications they missed. Since Redis Pub/Sub doesn't store messages, we rely on the PostgreSQL `notifications` table.

This is why we **always** save to PostgreSQL first — it's our source of truth.

In [ ]:
# Simulate: Mia opens the app and checks for unread notifications

print("📱 Mia opens the app and checks for notifications...\n")

unread = notifier.get_unread_notifications(4)

if unread:
    print(f"🔔 {len(unread)} unread notification(s):\n")
    for n in unread:
        print(f"   [{n['type'].upper()}] {n['message']}")
        print(f"   Received: {n['created_at']}")
        print()
    
    # Mark all as read
    ids = [n['id'] for n in unread]
    notifier.mark_as_read(ids)
    print(f"   ✅ Marked {len(ids)} notification(s) as read")
else:
    print("   No unread notifications")

# Verify they're marked as read
still_unread = notifier.get_unread_notifications(4)
print(f"\n   Unread count after marking: {len(still_unread)}")

durable = query("SELECT COUNT(*) AS cnt FROM notifications WHERE user_id = 4")[0]['cnt']
assert durable >= 1, (
    "Mia should have a durable notification row from the match above — writing to "
    "PostgreSQL before publishing is the only reason an offline user ever finds out."
)
assert len(still_unread) == 0, "everything we just marked as read should be read"


## 📊 Pub/Sub: Online vs Offline Detection

How do we know if a user is online? Redis `PUBLISH` returns the **number of subscribers** on that channel. If it returns 0, nobody is listening and we should send a push notification instead.

Let's see this in action:

In [ ]:
# Demonstrate online vs offline detection

print("📊 Pub/Sub Online Detection Demo\n")

# Test 1: Publish to a user who is NOT listening
offline_listeners = r.publish("notifications:user:999", json.dumps({"test": True}))
print(f"   User 999 (nobody listening): {offline_listeners} listener(s)")
print(f"   → Would send push notification (APNS/FCM)")
print()

# Test 2: Start a listener, then publish
def temp_listener(duration: float = 2.0):
    lr = redis.Redis(**REDIS_CONFIG)
    ps = lr.pubsub(ignore_subscribe_messages=True)
    ps.subscribe("notifications:user:999")
    deadline = time.time() + duration
    while time.time() < deadline:
        if ps.get_message(timeout=0.1):
            break
    ps.unsubscribe()
    ps.close()
    lr.close()

t = threading.Thread(target=temp_listener, daemon=True)
t.start()
assert wait_for_subscribers("notifications:user:999", 1), "the test listener never subscribed"

online_listeners = r.publish("notifications:user:999", json.dumps({"test": True}))
print(f"   User 999 (now listening): {online_listeners} listener(s)")
print(f"   → Delivered via Pub/Sub in real-time")

t.join(timeout=3)
print()
print("💡 PUBLISH returns the number of subscribers who received the message.")
print("   0 = offline → send push notification")
print("   1+ = online → delivered in real-time")

assert offline_listeners == 0, (
    f"Nobody should be listening on notifications:user:999 before we subscribe, "
    f"but PUBLISH reported {offline_listeners}. A leaked subscriber from an earlier "
    f"run would make the offline branch untestable."
)
assert online_listeners == 1, (
    f"Exactly one subscriber should be attached, PUBLISH reported {online_listeners}"
)
assert not t.is_alive(), "the test listener should have exited on its own"


## 🏗️ Production Architecture: Push Notifications

In production, Tinder uses device-native push notification services for offline users:

```
┌─────────────────────────────────────────────────────────────┐
│                     MATCH DETECTED                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. Save notification to PostgreSQL (source of truth)       │
│                                                             │
│  2. Publish to Redis Pub/Sub                                │
│     ├── listeners > 0 → Delivered! ✅                       │
│     └── listeners = 0 → User is offline                     │
│                                                             │
│  3. Send push notification                                  │
│     ├── iOS  → Apple Push Notification Service (APNS)       │
│     └── Android → Firebase Cloud Messaging (FCM)            │
│                                                             │
│  4. User opens app → Fetch unread from PostgreSQL           │
│     └── Subscribe to Pub/Sub for real-time updates          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### Why Redis Pub/Sub + Push Notifications?

| Feature | Redis Pub/Sub | Push (APNS/FCM) |
|---------|--------------|------------------|
| Latency | < 1ms | 100ms–seconds |
| Works offline? | No | Yes |
| Message storage | No (fire-and-forget) | Queued until delivered |
| Rich display | In-app only | Lock screen, badge, sound |

**Both are needed**: Pub/Sub for instant in-app updates, push for offline reach.

## ⚡ Bonus: Redis Pub/Sub Performance

Let's measure how fast Redis Pub/Sub delivers messages. At Tinder's scale with millions of matches per day, this needs to be fast.

In [ ]:
# Benchmark Redis Pub/Sub latency

latencies = []
received_count = 0

def benchmark_listener(count: int, timeout: float = 20.0):
    """Listen for 'count' messages and record when they arrive."""
    global received_count
    lr = redis.Redis(**REDIS_CONFIG)
    ps = lr.pubsub(ignore_subscribe_messages=True)
    ps.subscribe("benchmark:channel")
    
    deadline = time.time() + timeout
    while received_count < count and time.time() < deadline:
        msg = ps.get_message(timeout=0.1)
        if msg and msg['type'] == 'message':
            data = json.loads(msg['data'])
            latencies.append((time.time() - data['sent_at']) * 1000)  # ms
            received_count += 1
    
    ps.unsubscribe()
    ps.close()
    lr.close()

# Start listener
num_messages = 500
listener = threading.Thread(target=benchmark_listener, args=(num_messages,), daemon=True)
listener.start()
assert wait_for_subscribers("benchmark:channel", 1), "the benchmark listener never subscribed"

# Publish messages
print(f"⏱️  Publishing {num_messages} messages via Redis Pub/Sub...")
start = time.time()
for i in range(num_messages):
    r.publish("benchmark:channel", json.dumps({
        "id": i,
        "sent_at": time.time()
    }))
pub_time = time.time() - start

listener.join(timeout=25)

assert len(latencies) == num_messages, (
    f"Only {len(latencies)}/{num_messages} messages arrived. Pub/Sub is at-most-once, "
    f"but a subscriber attached before the first PUBLISH should not lose any — "
    f"dropped messages here mean the listener was too slow or died."
)

latencies.sort()
avg = sum(latencies) / len(latencies)
p50 = latencies[len(latencies) // 2]
p95 = latencies[int(len(latencies) * 0.95)]
p99 = latencies[int(len(latencies) * 0.99)]

print(f"\n📊 Pub/Sub Latency ({len(latencies)} messages):")
print(f"   Average: {avg:.3f} ms")
print(f"   P50:     {p50:.3f} ms")
print(f"   P95:     {p95:.3f} ms")
print(f"   P99:     {p99:.3f} ms")
print(f"   Publish throughput: {num_messages/pub_time:.0f} msg/sec")
print()
# Report what we actually measured rather than asserting a number in prose.
if p50 < 1:
    print("💡 Sub-millisecond delivery — everything happens in memory with no")
    print("   disk I/O, over a loopback connection.")
else:
    print(f"💡 P50 delivery was {p50:.2f} ms here — still in-memory, but the")
    print("   publisher and subscriber are Python threads fighting for the GIL.")
print("   In production, add the real network RTT between the phone and Redis.")

assert p99 < 250, (
    f"P99 delivery of {p99:.1f} ms is nowhere near the in-memory latency this cell "
    f"claims to demonstrate."
)


## 🧹 Cleanup

In [ ]:
# Clean up Redis keys
keys = r.keys("swipe:*") + r.keys("notifications:*") + r.keys("benchmark:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")

# Reset PostgreSQL to seed data
execute("DELETE FROM notifications WHERE id > 6")  # keep seed notifications
execute("DELETE FROM matches WHERE id > 3")         # keep seed matches
execute("DELETE FROM swipes WHERE id > 13")          # keep seed swipes
print("🧹 Reset PostgreSQL to seed data")

## 📚 Summary

### Key Takeaways

1. **Redis Pub/Sub** delivers messages in real-time to online users — sub-millisecond latency
2. **Pub/Sub is fire-and-forget** — if nobody is subscribed, the message is gone; always persist to the DB first
3. **Online detection** — `PUBLISH` returns the number of listeners; 0 means offline → send push notification
4. **Push notifications (APNS/FCM)** handle offline users — device-native, works even when the app is closed
5. **For durable real-time**, use **Redis Streams** (consumer groups, message replay) or **Kafka** instead of raw Pub/Sub
6. **Notify once per match, not once per request** — the swipe script suppresses replayed swipes, and `INSERT ... ON CONFLICT DO NOTHING RETURNING id` tells you whether *you* created the match. Only the creator notifies
7. **The full flow**: Save to DB → try Pub/Sub → fall back to push → user catches up on open

### The Complete Tinder Architecture (all 3 notebooks)

```
User opens app
  → Subscribe to Redis Pub/Sub (notifications:user:{id})
  → Fetch unread notifications from PostgreSQL
  → Load swipe stack via PostGIS query (Notebook 1)

User swipes right
  → Redis Lua script: atomic check + match detection (Notebook 2)
  → Persist swipe to PostgreSQL
  → If match:
      → Save match to PostgreSQL
      → Save notification to PostgreSQL
      → Publish via Redis Pub/Sub (Notebook 3)
      → Send push notification if offline
```

### What We Built Across All 3 Notebooks

| Component | Technology | Purpose |
|-----------|-----------|----------|
| Geolocation matching | PostGIS (GIST index) | Find nearby users with spatial indexes |
| Atomic swipe + match | Redis Lua scripts | Race-condition-free match detection |
| Real-time notifications | Redis Pub/Sub | Instant delivery to online users |
| Durable notifications | PostgreSQL | Catch-up when user was offline |
| Offline delivery | APNS / FCM (concept) | Reach users with the app closed |

🎉 **Congratulations!** You've built the core of a Tinder-like system from scratch.
